# LangGraph with AgentCore Memory using Episodic Strategy

# LangGraph 与 AgentCore Memory 使用情景策略

## Introduction

## 简介

This notebook demonstrates how to integrate Amazon Bedrock AgentCore Memory with **episodic memory strategy** in a conversational AI agent using LangGraph framework. We'll focus on the episodic strategy that captures complete conversation sessions, enabling the agent to recall specific meal planning episodes and track how dietary patterns evolve over time.

本笔记本演示如何在使用 LangGraph 框架的对话式 AI 代理中将 Amazon Bedrock AgentCore Memory 与**情景记忆策略**集成。我们将专注于捕获完整对话会话的情景策略，使代理能够回忆特定的膳食计划情景并跟踪饮食模式随时间的演变。

## Tutorial Details

## 教程详情

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Long-term Conversational                                                        |
| Agent usecase       | Nutrition Assistant with Episodic Memory Strategy                               |
| Agentic Framework   | LangGraph                                                                        |
| LLM model           | Anthropic Claude Sonnet 3.7                                                     |
| Tutorial components | AgentCore Memory, Episodic Strategy, LangGraph Hooks, Session-based Episodes  |
| Example complexity  | Intermediate                                                                     |

| 信息                 | 详情                                                                              |
|:--------------------|:---------------------------------------------------------------------------------|
| 教程类型             | 长期对话                                                                          |
| 代理用例             | 带有情景记忆策略的营养助手                                                          |
| 代理框架             | LangGraph                                                                        |
| LLM 模型             | Anthropic Claude Sonnet 3.7                                                      |
| 教程组件             | AgentCore Memory、情景策略、LangGraph 钩子、基于会话的情景                          |
| 示例复杂度           | 中级                                                                              |

You'll learn to:

您将学习：

- Create AgentCore Memory with episodic memory strategy
- Implement pre/post model hooks for automatic memory storage
- Build a nutrition assistant that remembers meal planning sessions
- Retrieve and reflect on past conversations
- Track dietary patterns over time

- 使用情景记忆策略创建 AgentCore Memory
- 实现用于自动记忆存储的模型前/后钩子
- 构建一个记住膳食计划会话的营养助手
- 检索和反思过去的对话
- 随时间跟踪饮食模式

### Scenario Context

### 场景背景

In this example, we'll create a **Nutrition Assistant** that remembers complete meal planning sessions using episodic memory strategy. The agent will capture full conversation episodes including recipe discussions, ingredient substitutions, and meal feedback. This enables temporal queries like "What did I plan last week?" and pattern analysis of dietary habits.

在本示例中，我们将创建一个使用情景记忆策略记住完整膳食计划会话的**营养助手**。代理将捕获完整的对话情景，包括食谱讨论、食材替换和膳食反馈。这使得诸如"我上周计划了什么？"之类的时间性查询以及饮食习惯的模式分析成为可能。

## Architecture

## 架构

<div style="text-align:left">
    <img src="architecture_episodic.png" width="65%" />
</div>

### Why Episodic Memory Strategy for Nutrition?

### 为什么营养场景适合使用情景记忆策略？

- **Session-based**: Each meal planning conversation is an episode
- **Temporal context**: Meals are tied to specific times/occasions
- **Pattern learning**: Track how preferences evolve
- **Rich recall**: Remember full context of past recommendations

- **基于会话**：每次膳食计划对话都是一个情景
- **时间上下文**：膳食与特定时间/场合相关联
- **模式学习**：跟踪偏好如何演变
- **丰富回忆**：记住过去建议的完整上下文

### How Episodic Memory Strategy Works

### 情景记忆策略如何工作

The episodic strategy is designed to capture interactions as structured episodes and reflect across these episodes to generate meaningful insights. This strategy records not only what happened, but also the intent, thoughts, and outcome for each episode.

情景策略旨在将交互捕获为结构化情景，并跨这些情景进行反思以生成有意义的洞察。此策略不仅记录发生了什么，还记录每个情景的意图、想法和结果。

#### Three Steps in Episodic Strategy:

#### 情景策略的三个步骤：

1. **Extraction** – Identifies useful insights from short-term memory to place into long-term memory as memory records
2. **Consolidation** – Determines whether to write useful information to a new record or an existing record
3. **Reflection** – Insights are generated across episodes from agent interactions

1. **提取** – 从短期记忆中识别有用的洞察并作为记忆记录放入长期记忆
2. **整合** – 确定将有用信息写入新记录还是现有记录
3. **反思** – 从代理交互的情景中生成洞察

#### Strategy Output:

#### 策略输出：

**Episodes** (XML-formatted):

**情景**（XML 格式）：

- Broken down into: situation, intent, assessment, justification, and episode-level reflection
- Analyzed turn-by-turn as the interaction proceeds
- Helps understand order of operations and tool use

- 分解为：情境、意图、评估、理由和情景级反思
- 随着交互进行逐轮分析
- 有助于理解操作顺序和工具使用

**Reflections** (generated in background):

**反思**（在后台生成）：

- Consolidate across multiple episodes
- Extract broader insights identifying:
  - Successful strategies and patterns
  - Potential improvements
  - Common failure modes
  - Lessons learned spanning multiple interactions

- 跨多个情景整合
- 提取更广泛的洞察，识别：
  - 成功的策略和模式
  - 潜在的改进
  - 常见的失败模式
  - 跨多个交互的经验教训

#### For Nutrition Assistant:

#### 对于营养助手：

- **Episodes**: Each meal planning session (recipes discussed, ingredients, decisions)
- **Reflections**: Dietary patterns, favorite cuisines, cooking skill progression
- **Turn-by-turn**: Recipe exploration → ingredient questions → substitutions → final choice

- **情景**：每次膳食计划会话（讨论的食谱、食材、决策）
- **反思**：饮食模式、喜欢的菜系、烹饪技能进展
- **逐轮**：食谱探索 → 食材问题 → 替换 → 最终选择

## Prerequisites

## 前提条件

- Python 3.10+
- AWS account with appropriate permissions
- AWS IAM role with appropriate permissions for AgentCore Memory
- Access to Amazon Bedrock models

- Python 3.10+
- 具有适当权限的 AWS 账户
- 具有 AgentCore Memory 适当权限的 AWS IAM 角色
- 访问 Amazon Bedrock 模型

Let's get started by setting up our environment!

让我们开始设置环境吧！

In [ ]:
# Install necessary libraries from https://github.com/langchain-ai/langchain-aws
%pip install -qr requirements.txt

In [ ]:
import os
import logging

# Import LangGraph and LangChain components
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableConfig
from langgraph.store.base import BaseStore
import uuid


region = os.getenv("AWS_REGION", "us-east-1")
logging.getLogger("nutrition-agent").setLevel(logging.DEBUG)

In [ ]:
# Import the AgentCoreMemoryStore that we will use as a store
from langgraph_checkpoint_aws import AgentCoreMemoryStore

# For this example, we will just use an InMemorySaver to save context.
# In production, we highly recommend the AgentCoreMemorySaver as a checkpointer which works seamlessly alongside the memory store
# from langgraph_checkpoint_aws import AgentCoreMemorySaver
from langgraph.checkpoint.memory import InMemorySaver
from bedrock_agentcore.memory import MemoryClient

In [ ]:
import boto3
import json

# Create IAM role for memory execution
iam_client = boto3.client("iam")
sts_client = boto3.client("sts")
account_id = sts_client.get_caller_identity()["Account"]

ROLE_NAME = "AgentCoreMemoryExecutionRole"

# Trust policy for AgentCore Memory (gamma endpoints)
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": [
                    "preprod.genesis-service.aws.internal",
                    "bedrock-agentcore.amazonaws.com",
                    "developer.genesis-service.aws.internal",
                ]
            },
            "Action": "sts:AssumeRole",
        }
    ],
}

# Permissions for Bedrock model invocation
permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
            "Resource": [
                "arn:aws:bedrock:*::foundation-model/*",
                "arn:aws:bedrock:*:*:inference-profile/*",
            ],
        }
    ],
}

try:
    # Try to get existing role
    role = iam_client.get_role(RoleName=ROLE_NAME)
    MEMORY_EXECUTION_ROLE_ARN = role["Role"]["Arn"]
    print(f"✅ Using existing role: {MEMORY_EXECUTION_ROLE_ARN}")
except iam_client.exceptions.NoSuchEntityException:
    # Create role
    print(f"Creating IAM role: {ROLE_NAME}")
    role = iam_client.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Execution role for AgentCore Memory with custom strategies",
    )
    MEMORY_EXECUTION_ROLE_ARN = role["Role"]["Arn"]

    # Attach inline policy
    iam_client.put_role_policy(
        RoleName=ROLE_NAME,
        PolicyName="BedrockModelAccess",
        PolicyDocument=json.dumps(permissions_policy),
    )
    print(f"✅ Created role: {MEMORY_EXECUTION_ROLE_ARN}")
    print("⏳ Waiting 10 seconds for IAM propagation...")
    import time

    time.sleep(10)

print(f"\nRole ARN: {MEMORY_EXECUTION_ROLE_ARN}")

In [ ]:
memory_name = "NutritionAssistantEpisodic"
client = MemoryClient(region_name=region)
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"

override_strategy = {
    "customMemoryStrategy": {
        "name": "NutritionEpisodicExtractor",
        "description": "Nutrition assistant with episodic memory for meal planning insights",
        "namespaces": ["nutrition/{actorId}/{sessionId}"],
        "configuration": {
            "episodicOverride": {
                "extraction": {
                    "modelId": MODEL_ID,
                    "appendToPrompt": "Extract meal planning conversations including recipes discussed, ingredients mentioned, dietary considerations, and user feedback.",
                },
                "consolidation": {
                    "modelId": MODEL_ID,
                    "appendToPrompt": "Consolidate meal planning sessions into episodes, capturing the flow of recipe exploration and decision-making.",
                },
                "reflection": {
                    "modelId": MODEL_ID,
                    "appendToPrompt": "Generate insights about dietary patterns, favorite recipes, and how meal preferences evolve over time.",
                    "namespaces": ["nutrition/{actorId}"],
                },
            }
        },
    }
}

memory = client.create_or_get_memory(
    name=memory_name,
    description="Nutrition assistant with episodic memory for meal planning sessions",
    memory_execution_role_arn=MEMORY_EXECUTION_ROLE_ARN,
    strategies=[override_strategy],
)
memory_id = memory["id"]

print(f"✅ Created episodic memory: {memory_id}")

### Memory Configuration Overview

### 记忆配置概述

Our AgentCore Episodic Memory setup includes:

我们的 AgentCore 情景记忆设置包括：

- **Extraction**: Captures meal planning conversations with recipes, ingredients, and feedback
- **Consolidation**: Groups conversations into meal planning episodes
- **Reflection**: Generates insights about dietary patterns and preferences over time
- **Namespaces**: Organizes episodes by user (`nutrition/{actorId}`)

- **提取**：捕获包含食谱、食材和反馈的膳食计划对话
- **整合**：将对话分组为膳食计划情景
- **反思**：生成关于饮食模式和偏好随时间变化的洞察
- **命名空间**：按用户组织情景（`nutrition/{actorId}`）

Each conversation session becomes an episode that can be recalled and analyzed.

每个对话会话都成为可以被回忆和分析的情景。

## Step 3: Initialize Memory Store and LLM

## 第三步：初始化记忆存储和 LLM

Now we'll initialize the AgentCore Memory Store and our language model.

现在我们将初始化 AgentCore Memory Store 和我们的语言模型。

In [ ]:
# Initialize the store to enable long term memory saving and retrieval
store = AgentCoreMemoryStore(memory_id=memory_id, region_name=region)

# Initialize Bedrock LLM
llm = init_chat_model(MODEL_ID, model_provider="bedrock_converse", region_name=region)

## Step 4: Implement Memory Hooks

## 第四步：实现记忆钩子

We'll create pre and post model hooks to automatically handle memory storage:

我们将创建模型前和模型后钩子来自动处理记忆存储：

- **Pre-model hook**: Saves the user message before LLM invocation
- **Post-model hook**: Saves the assistant response after LLM invocation

- **模型前钩子**：在 LLM 调用前保存用户消息
- **模型后钩子**：在 LLM 调用后保存助手响应

### How Memory Processing Works

### 记忆处理工作原理

1. Messages are saved to AgentCore Memory with actor_id and session_id
2. The episodic strategy processes conversations to create structured episodes
3. Episodes are stored in the `nutrition/{actorId}/{sessionId}` namespace with turn-by-turn analysis
4. Reflections are generated across episodes and stored in the `nutrition/{actorId}` namespace
5. Each episode captures situation, intent, assessment, and conversation flow

1. 消息通过 actor_id 和 session_id 保存到 AgentCore Memory
2. 情景策略处理对话以创建结构化情景
3. 情景存储在 `nutrition/{actorId}/{sessionId}` 命名空间中，带有逐轮分析
4. 反思跨情景生成并存储在 `nutrition/{actorId}` 命名空间中
5. 每个情景捕获情境、意图、评估和对话流程

**Note**: LangChain message types are converted under the hood by the store to AgentCore Memory message types so that they can be properly processed into episodes and reflections.

**注意**：LangChain 消息类型在底层被存储转换为 AgentCore Memory 消息类型，以便可以正确处理为情景和反思。

In [ ]:
def pre_model_hook(state, config: RunnableConfig, *, store: BaseStore):
    """Hook that runs pre-LLM invocation to save the latest human message"""
    actor_id = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]
    # Saving the message to the actor and session combination that we get at runtime
    namespace = (actor_id, thread_id)

    messages = state.get("messages", [])
    # Save the last human message we see before LLM invocation
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            break

    # For episodic strategy, we just save messages - no retrieval needed
    # Episodes and reflections are generated automatically in the background
    return {"messages": messages}


def post_model_hook(state, config: RunnableConfig, *, store: BaseStore):
    """Hook that runs post-LLM invocation to save the assistant response"""
    actor_id = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]

    # Saving the message to the actor and session combination that we get at runtime
    namespace = (actor_id, thread_id)

    messages = state.get("messages", [])
    # Save the LLM's response to AgentCore Memory
    for msg in reversed(messages):
        if isinstance(msg, AIMessage):
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            break

    return {"messages": messages}

## Step 5: Create the LangGraph Agent

## 第五步：创建 LangGraph 代理

Now we'll create our nutrition assistant agent using LangGraph's `create_react_agent` with our memory hooks integrated. The tool node will contain just our long term memory retrieval tool and the pre and post model hooks are specified as arguments.

现在我们将使用 LangGraph 的 `create_react_agent` 创建我们的营养助手代理，并集成我们的记忆钩子。工具节点将只包含我们的长期记忆检索工具，模型前和模型后钩子作为参数指定。

**Note**: for custom agent implementations the Store and tools can be configured to run as needed for any workflow following this pattern. Pre/post model hooks can be used, the whole conversation could be saved at the end, etc.

**注意**：对于自定义代理实现，Store 和工具可以按照此模式为任何工作流配置为按需运行。可以使用模型前/后钩子，也可以在最后保存整个对话等。

In [ ]:
graph = create_react_agent(
    llm,
    store=store,
    tools=[],  # No additional tools needed for this example
    checkpointer=InMemorySaver(),  # For conversation state management
    pre_model_hook=pre_model_hook,  # Saves user message before LLM call
    post_model_hook=post_model_hook,  # Saves assistant response for episodic processing after LLM call
)

## Step 6: Configure Agent Runtime

## 第六步：配置代理运行时

We need to configure the agent with unique identifiers for the user and session. These IDs are crucial for memory organization and retrieval.

我们需要为用户和会话配置代理的唯一标识符。这些 ID 对于记忆组织和检索至关重要。

### Graph Invoke Input

### 图调用输入

We only need to pass the newest user message in as an argument `inputs`. This could include other state variables as well but for the simple `create_react_agent`, we only need messages.

我们只需要将最新的用户消息作为参数 `inputs` 传入。这也可以包含其他状态变量，但对于简单的 `create_react_agent`，我们只需要消息。

### LangGraph RuntimeConfig

### LangGraph 运行时配置

In LangGraph, config is a `RuntimeConfig` that contains attributes that are necessary at invocation time, for example user IDs or session IDs. For the `AgentCoreMemorySaver`, `thread_id` and `actor_id` must be set in the config. For instance, your AgentCore invocation endpoint could assign this based on the identity or user ID of the caller. You can read additional [documentation here](https://langchain-ai.github.io/langgraphjs/how-tos/configuration/)

在 LangGraph 中，config 是一个 `RuntimeConfig`，包含调用时必需的属性，例如用户 ID 或会话 ID。对于 `AgentCoreMemorySaver`，必须在配置中设置 `thread_id` 和 `actor_id`。例如，您的 AgentCore 调用端点可以根据调用者的身份或用户 ID 来分配这些值。您可以在[此处阅读更多文档](https://langchain-ai.github.io/langgraphjs/how-tos/configuration/)

In [ ]:
actor_id = "user-1"
config = {
    "configurable": {
        "thread_id": "session-1",  # REQUIRED: This maps to Bedrock AgentCore session_id under the hood
        "actor_id": actor_id,  # REQUIRED: This maps to Bedrock AgentCore actor_id under the hood
    }
}

## Step 7: Test the Agent

## 第七步：测试代理

Let's test our nutrition assistant by having a conversation about food preferences. The agent will automatically capture the conversation as episodes for future recall and pattern analysis.

让我们通过一次关于食物偏好的对话来测试我们的营养助手。代理将自动将对话捕获为情景，以供将来回忆和模式分析。

In [ ]:
# Helper function to pretty print agent output while running
def run_agent(query: str, config: RunnableConfig):
    printed_ids = set()
    events = graph.stream(
        {"messages": [{"role": "user", "content": query}]},
        config,
        stream_mode="values",
    )
    for event in events:
        if "messages" in event:
            for msg in event["messages"]:
                # Check if we've already printed this message
                if id(msg) not in printed_ids:
                    msg.pretty_print()
                    printed_ids.add(id(msg))


prompt = """
Hey there! Im cooking one of my favorite meals tonight, salmon with rice and veggies (healthy). Has
great macros for my weightlifting competition that is coming up. What can I add to this dish to make it taste better
and also improve the protein and vitamins I get?
"""

run_agent(prompt, config)

### What was stored?

### 存储了什么？

As you can see, the model does not yet have any insights from previous meal planning sessions.

如您所见，模型尚未从以前的膳食计划会话中获得任何洞察。

For this implementation with pre/post model hooks, two messages were stored here. The first message from the user and the response from the AI model were both stored as conversational events in AgentCore Memory. It may take a few moments for the episodes and reflections to be generated, so retry after a few mins if nothing is found the first try.

对于这个使用模型前/后钩子的实现，这里存储了两条消息。来自用户的第一条消息和来自 AI 模型的响应都作为对话事件存储在 AgentCore Memory 中。情景和反思的生成可能需要一些时间，所以如果第一次没有找到任何内容，请几分钟后重试。

These messages were then processed by the episodic strategy to create structured episodes and reflections in AgentCore long term memory. In fact, we can check the store ourselves to verify what has been stored there so far:

这些消息随后被情景策略处理，在 AgentCore 长期记忆中创建结构化情景和反思。事实上，我们可以自己检查存储来验证到目前为止存储了什么：

In [ ]:
# Search our conversation messages
search_namespace = ("nutrition", actor_id, "session-1")
result = store.search(search_namespace, query="meal", limit=3)
print(f"Conversation messages result: {result}")

In [ ]:
# The correct way to search episodic long-term memories in LangGraph
from bedrock_agentcore.memory import MemoryClient

# Use the memory client directly (not the store)
memory_client = MemoryClient(region_name=region)

print("=== Searching Long-Term Episodic Memories ===")
print(f"Memory ID: {memory_id}")
print()

# Search episodic memories (episodes)
print("1. Episodic namespace: nutrition/user-1/session-1")
try:
    episodes = memory_client.retrieve_memories(
        memory_id=memory_id,
        namespace="nutrition/user-1/session-1",
        query="meal",
        top_k=3,
    )
    print(f"   Found {len(episodes)} episode memories")
    for mem in episodes:
        content = mem.get("content", {})
        text = content.get("text", str(content))
        print(f"   - {text[:300]}...")
except Exception as e:
    print(f"   Error: {e}")
print()

# Search reflection memories
print("2. Reflection namespace: nutrition/user-1")
try:
    reflections = memory_client.retrieve_memories(
        memory_id=memory_id, namespace="nutrition/user-1", query="meal", top_k=3
    )
    print(f"   Found {len(reflections)} reflection memories")
    for mem in reflections:
        content = mem.get("content", {})
        text = content.get("text", str(content))
        print(f"   - {text[:300]}...")
except Exception as e:
    print(f"   Error: {e}")

### Agent access to the store

### 代理访问存储

**Note** - since AgentCore memory processes these events in the background, it may take a few mins for the memory to be extracted and embedded to long term memory retrieval.

**注意** - 由于 AgentCore memory 在后台处理这些事件，记忆的提取和嵌入到长期记忆检索可能需要几分钟。

Great! Now we have seen that long term memories were extracted to our namespaces based on the earlier messages in the conversation.

很好！现在我们已经看到长期记忆根据对话中的早期消息被提取到我们的命名空间中。

Now, let's start a new session and ask about recommendations for what to cook for dinner. The agent can use the store to access the long term memories that were extracted to make a recommendation that the user will be sure to like.

现在，让我们开始一个新会话并询问晚餐做什么的建议。代理可以使用存储来访问被提取的长期记忆，以做出用户肯定会喜欢的推荐。

In [ ]:
config = {
    "configurable": {
        "thread_id": "session-2",  # New session ID
        "actor_id": actor_id,  # Same actor ID
    }
}

run_agent("Today's a new day, what should I make for dinner tonight?", config)

### Wrapping up

### 总结

As you can see, the agent's conversations are automatically captured and processed into structured episodes with turn-by-turn analysis. The episodic strategy generates insights across multiple meal planning sessions to identify patterns and track how preferences evolve over time.

如您所见，代理的对话被自动捕获并处理为带有逐轮分析的结构化情景。情景策略跨多个膳食计划会话生成洞察，以识别模式并跟踪偏好随时间的演变。

The AgentCoreMemoryStore is very flexible and can be implemented in a variety of ways, including pre/post model hooks or just tools themselves with store operations. Used alongside the AgentCoreMemorySaver for checkpointing, both full conversational state and episodic reflections can be combined to form a complex and intelligent agent system.

AgentCoreMemoryStore 非常灵活，可以通过多种方式实现，包括模型前/后钩子或仅使用带有存储操作的工具本身。与用于检查点的 AgentCoreMemorySaver 一起使用，完整的对话状态和情景反思都可以结合起来形成一个复杂而智能的代理系统。